# Predictive Modeling: Air Pollution, Mortality, and Population

This notebook develops and compares several machine learning models to predict mortality rates based on air pollution exposure and population data. The workflow includes data preprocessing, feature engineering, model training, evaluation, and interpretation.

## Model Selection Rationale

We selected a diverse set of models to capture both linear and non-linear relationships:

- **Linear Regression, Ridge, Lasso:** Baseline models for interpretability and to assess linear relationships.
- **Random Forest:** A robust tree-based ensemble model, effective for non-linear patterns and feature interactions.
- **XGBoost:** Gradient boosting, known for high accuracy and handling complex data structures.
- **CatBoost:** Handles categorical features natively and is resilient to overfitting.

This mix allows us to compare simple and advanced approaches, ensuring the best fit for our data's complexity.

In [1]:
!pip3 install lightgbm xgboost catboost scikit-learn pandas matplotlib seaborn



[notice] A new release of pip is available: 25.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


## 1. Install Required Packages
This cell installs all necessary Python packages for modeling, data manipulation, and visualization. If running in a new environment, ensure these libraries are installed to avoid import errors.

## 2. Import Libraries and Load Data

Importing essential libraries for data manipulation, analysis, and machine learning model development.

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

### Importing Libraries
This cell imports essential libraries for data manipulation (pandas, numpy), visualization (plotly, matplotlib, seaborn), and machine learning (scikit-learn, XGBoost, CatBoost). These tools provide a comprehensive environment for analysis and modeling.

In [3]:
try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
    print("XGBoost is available")
except Exception:
    HAVE_XGB = False

try:
    from catboost import CatBoostRegressor
    HAVE_CAT = True
    print("CatBoost is available")
except Exception:
    HAVE_CAT = False

XGBoost is available
CatBoost is available


### Checking for XGBoost and CatBoost Availability
This cell checks if XGBoost and CatBoost libraries are installed and available. These advanced models are included for their ability to handle complex, non-linear relationships and categorical features.

## 3. Data Exploration and Initial Assessment

Loading the final merged dataset that combines air pollution exposure data with mortality and population statistics. This dataset represents the culmination of extensive data preprocessing and integration efforts.

In [4]:
df = pd.read_csv("predictive_modeling_data_prepared.csv")
print(df.shape)
df.head()

(189720, 11)


,Cause,Country,Year,Deaths,Region Name,HAP,NO2,OZONE,PM25,Population,Mortality Rate
0,Acute hepatitis A,Argentina,1990,22.093804,Southern Latin America,0.10,18.90,33.9,19.0,32755901.0,0.067450
1,Acute hepatitis E,Argentina,1990,0.698845,Southern Latin America,0.10,18.90,33.9,19.0,32755901.0,0.002133
2,Rheumatoid arthritis,Myanmar,1990,58.031486,Southeast Asia,0.96,5.14,35.3,41.6,39817251.0,0.145745
3,Drowning,Guyana,1990,66.605160,Caribbean,0.19,3.46,21.9,24.3,749894.0,8.881943
4,Drowning,Mauritania,1990,131.445606,Western Africa,0.85,5.26,33.4,78.6,1951878.0,6.734315


### Load and Preview Data
This cell loads the prepared dataset containing air pollution, mortality, and population data. It displays the shape and a preview of the data to verify successful loading and initial structure.

In [5]:
df = df[df["Population"] > 0].copy()
df = df.dropna(subset=["Population", "Deaths"])

### Data Cleaning: Remove Invalid Entries
This cell filters out rows with missing or zero population and deaths, ensuring the dataset is suitable for modeling and free from invalid values.

In [6]:
features = [
    "HAP",
    "NO2",
    "OZONE",
    "PM25",
    "Population",
    "Year",
    "Region Name",
    "Cause"
]

### Select Features for Modeling
This cell defines the list of features to be used in the predictive models, including pollutant levels, population, year, region, and cause of death.

## 6. Feature Engineering: Log Transformation

### Rationale for Log Transformation
Applying log1p (log(1+x)) transformation to handle:
1. **Skewed Distributions**: Air pollution and population data often exhibit right-skewed distributions
2. **Scale Normalization**: Reducing the impact of extreme values
3. **Model Stability**: Improving numerical stability for gradient-based algorithms
4. **Interpretability**: Log-transformed coefficients represent percentage changes

This transformation is particularly important for tree-based models and linear models, helping to capture non-linear relationships and reducing the influence of outliers.

In [7]:
df["log_Population"] = np.log1p(df["Population"])
df["log_PM25"] = np.log1p(df["PM25"])
df["log_NO2"] = np.log1p(df["NO2"])
df["log_OZONE"] = np.log1p(df["OZONE"])
df["log_HAP"] = np.log1p(df["HAP"])

### Feature Engineering: Log Transformation
Log transformation is applied to skewed features (pollutants, population) to normalize distributions, reduce the impact of outliers, and improve model stability and interpretability.

In [8]:
TARGET = 'Mortality Rate'
df['log_target'] = np.log1p(df[TARGET].astype(float))

### Transform Target Variable
The target variable (Mortality Rate) is log-transformed to match the feature engineering and stabilize variance, making model predictions more robust.

In [9]:
num_features = ['log_PM25', 'log_NO2', 'log_OZONE', 'log_HAP', 'log_Population']
cat_features = ['Cause', 'Region Name'] 

### Define Numeric and Categorical Features
This cell separates features into numeric and categorical groups, which is essential for appropriate preprocessing and encoding in machine learning pipelines.

In [10]:
# Sanity check: ensure required columns exist
missing_cols = [c for c in (num_features + cat_features + ['Year', TARGET]) if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

### Sanity Check: Validate Required Columns
This cell ensures all necessary columns for modeling are present in the dataset, preventing errors during training and evaluation.

In [11]:
# 3) Temporal split: train on first ~80% of years, test on last ~20%
def temporal_train_test_split(frame, year_col='Year', test_size=0.2):
    years = np.sort(frame[year_col].unique())
    if len(years) < 3:
        raise ValueError("Need at least 3 unique years for a meaningful temporal split.")
    split_idx = int(np.ceil(len(years) * (1 - test_size)))
    train_years = years[:split_idx]
    test_years = years[split_idx:]
    train = frame[frame[year_col].isin(train_years)].copy()
    test  = frame[frame[year_col].isin(test_years)].copy()
    return train, test, train_years, test_years

### Temporal Train-Test Split
This cell splits the data into training and testing sets based on years, ensuring the model is evaluated on future (unseen) years for realistic performance assessment.

In [12]:
df_train, df_test, train_years, test_years = temporal_train_test_split(df, year_col='Year', test_size=0.2)
print(f"Train years: {train_years[0]}–{train_years[-1]} | Test years: {test_years[0]}–{test_years[-1]}")
print(f"Train size: {len(df_train):,} rows | Test size: {len(df_test):,} rows")


Train years: 1990–2014 | Test years: 2015–2020
Train size: 147,750 rows | Test size: 35,460 rows


### Display Train-Test Split Details
This cell prints the years and sizes of the training and test sets, confirming the temporal split and providing context for model evaluation.

In [13]:
unseen_causes = set(df_test['Cause'].unique()) - set(df_train['Cause'].unique())
unseen_regions = set(df_test['Region Name'].unique()) - set(df_train['Region Name'].unique())
if unseen_causes:
    print("⚠️  Unseen causes in test:", unseen_causes)
if unseen_regions:
    print("⚠️  Unseen regions in test:", unseen_regions)

### Check for Unseen Categories in Test Set
This cell identifies any causes or regions present in the test set but not in the training set, highlighting potential generalization challenges for the models.

In [14]:
X_train = df_train[num_features + cat_features].copy()
X_test  = df_test[num_features + cat_features].copy()
y_train = df_train['log_target'].values
y_test  = df_test['log_target'].values

### Prepare Training and Test Data
This cell creates the feature matrices and target vectors for both training and test sets, readying the data for model fitting.

In [15]:
# 5) Preprocessing:
#    - Standardize numeric features for linear/ridge/lasso stability
#    - One-hot encode categoricals; drop='first' avoids dummy trap for linear models

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = OneHotEncoder(handle_unknown='ignore', drop='first', sparse=True)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features),
    ],
    remainder='drop'
)


### Preprocessing: Scaling and Encoding
This cell sets up preprocessing pipelines to standardize numeric features and one-hot encode categorical features, ensuring compatibility and optimal performance for linear and tree-based models.

In [16]:
# Helper to evaluate models (trained to predict log(target))
def evaluate_and_report(name, pipeline, Xtr, ytr, Xte, yte):
    pipeline.fit(Xtr, ytr)
    y_pred_log = pipeline.predict(Xte)
    # invert log1p -> original scale for metrics users care about
    y_true = np.expm1(yte)
    y_pred = np.clip(np.expm1(y_pred_log), a_min=0.0, a_max=None)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {'model': name, 'RMSE': rmse, 'R2': r2, 'fitted_estimator': pipeline}


### Model Evaluation Function
This cell defines a helper function to train models, make predictions, and compute performance metrics (RMSE, R²) on the test set, enabling consistent comparison across models.

In [17]:

# 6) Define and train pipelines for 5 models that use OHE preprocessor
models = []

models.append((
    "LinearRegression",
    Pipeline(steps=[('prep', preprocessor), ('model', LinearRegression())])
))

models.append((
    "Ridge",
    Pipeline(steps=[('prep', preprocessor), ('model', Ridge(alpha=1.0))])
))

models.append((
    "Lasso",
    Pipeline(steps=[('prep', preprocessor), ('model', Lasso(alpha=1e-3, max_iter=20000))])
))

models.append((
    "RandomForest",
    Pipeline(steps=[('prep', preprocessor),
                    ('model', RandomForestRegressor(
                        n_estimators=400, max_depth=None, min_samples_leaf=2,
                        random_state=42, n_jobs=-1
                    ))])
))

models.append((
        "XGBoost",
        Pipeline(steps=[('prep', preprocessor),
                        ('model', XGBRegressor(
                            n_estimators=800,
                            max_depth=4,
                            learning_rate=0.05,
                            subsample=0.8,
                            colsample_bytree=0.8,
                            objective='reg:squarederror',
                            n_jobs=-1,
                            tree_method='hist',
                            reg_lambda=1.0
                        ))])
    ))

### Define and Train Models
This cell sets up pipelines for five models: Linear Regression, Ridge, Lasso, Random Forest, and XGBoost. Each pipeline includes preprocessing and model fitting, allowing direct comparison of their performance.

In [18]:
# 7) CatBoost: give raw categorical columns directly (better than OHE for CatBoost)
results = []
for name, pipe in models:
    res = evaluate_and_report(name, pipe, X_train, y_train, X_test, y_test)
    results.append(res)
    print(f"{name:>14} | RMSE={res['RMSE']:.4f} | R2={res['R2']:.4f}")



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


LinearRegression | RMSE=22.7685 | R2=0.4634
         Ridge | RMSE=22.7731 | R2=0.4632


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


         Lasso | RMSE=22.9985 | R2=0.4525


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


  RandomForest | RMSE=7.4737 | R2=0.9422


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


       XGBoost | RMSE=11.2750 | R2=0.8684


### Train and Evaluate Models
This cell trains each model pipeline and evaluates their performance on the test set, printing RMSE and R² scores for direct comparison.

In [19]:
# CatBoost can take a DataFrame with string categories; specify categorical feature names
X_train_cb = df_train[num_features + cat_features].copy()
X_test_cb  = df_test[num_features + cat_features].copy()

cb = CatBoostRegressor(
    iterations=1200,
    depth=6,
    learning_rate=0.05,
    loss_function='RMSE',
    random_seed=42,
    verbose=0
)
cb.fit(X_train_cb, y_train, cat_features=cat_features)
y_pred_log = cb.predict(X_test_cb)
y_true = np.expm1(y_test)
y_pred = np.clip(np.expm1(y_pred_log), 0.0, None)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
results.append({'model': 'CatBoost', 'RMSE': rmse, 'R2': r2, 'fitted_estimator': cb})
print(f"{'CatBoost':>14} | RMSE={rmse:.4f} | R2={r2:.4f}")

      CatBoost | RMSE=16.6624 | R2=0.7126


### Train and Evaluate CatBoost Model
CatBoost is trained using raw categorical features, leveraging its native handling of categorical data. Its performance is evaluated and compared to other models.

In [20]:
# 8) Results table
results_df = (pd.DataFrame(results)
                .drop(columns=['fitted_estimator'])
                .sort_values(by=['RMSE','R2'], ascending=[True, False])
                .reset_index(drop=True))
print("\n=== Test Set Performance (future years) ===")
print(results_df)


=== Test Set Performance (future years) ===
              model       RMSE        R2
0      RandomForest   7.473718  0.942181
1           XGBoost  11.275015  0.868408
2          CatBoost  16.662425  0.712611
3  LinearRegression  22.768488  0.463385
4             Ridge  22.773071  0.463169
5             Lasso  22.998540  0.452486


### Results Table: Model Performance
This cell summarizes the test set performance of all models in a table, sorted by RMSE and R², to facilitate comparison and selection of the best model.

In [21]:
pd.DataFrame(results_df)

,model,RMSE,R2
0,RandomForest,7.473718,0.942181
1,XGBoost,11.275015,0.868408
2,CatBoost,16.662425,0.712611
3,LinearRegression,22.768488,0.463385
4,Ridge,22.773071,0.463169
5,Lasso,22.998540,0.452486


### Display Results DataFrame
This cell displays the rounded results DataFrame for easier interpretation and reporting.

In [22]:
# 9) (Optional) inspect feature importances or coefficients for the best model with OHE:
#    Note: for tree-based models with OHE pipeline, you can get feature names like this:
def get_feature_names(prep: ColumnTransformer):
    num_names = list(prep.named_transformers_['num']\
                     .named_steps['scaler']\
                     .get_feature_names_out(num_features))
    cat_names = list(prep.named_transformers_['cat']\
                     .get_feature_names_out(cat_features))
    return num_names + cat_names

best_row = results_df.iloc[0]
best_name = best_row['model']
print(f"\nBest model by RMSE: {best_name}")



Best model by RMSE: RandomForest


### Feature Importance and Model Interpretation
This cell inspects the best model's coefficients or feature importances, providing insights into which features most strongly influence predictions.

In [23]:
# Example: if the best is a pipeline with tree/coefs, you can introspect like this:
best_estimator = [r['fitted_estimator'] for r in results if r['model'] == best_name][0]

if isinstance(best_estimator, Pipeline):
    prep = best_estimator.named_steps['prep']
    feat_names = get_feature_names(prep)
    model = best_estimator.named_steps['model']

    # Linear-family coefficients
    if hasattr(model, 'coef_'):
        coef = np.ravel(model.coef_)
        top_idx = np.argsort(np.abs(coef))[-20:][::-1]
        top_features = [(feat_names[i], coef[i]) for i in top_idx]
        print("\nTop 20 absolute coefficients:")
        for f, c in top_features:
            print(f"{f:45s} {c: .4f}")

    # Tree-based feature importances
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        top_idx = np.argsort(importances)[-20:][::-1]
        print("\nTop 20 feature importances:")
        for i in top_idx:
            print(f"{feat_names[i]:45s} {importances[i]:.5f}")

elif HAVE_CAT and isinstance(best_estimator, CatBoostRegressor):
    # CatBoost has its own feature importance API
    importances = best_estimator.get_feature_importance()
    all_features = num_features + cat_features
    order = np.argsort(importances)[-20:][::-1]
    print("\nTop 20 CatBoost feature importances:")
    for i in order:
        print(f"{all_features[i]:45s} {importances[i]:.5f}")


Top 20 feature importances:
Cause_Total cancers                           0.21689
log_HAP                                       0.15884
Cause_Chronic obstructive pulmonary disease   0.06784
Cause_Tracheal, bronchus, and lung cancer     0.04490
Cause_Alzheimer's disease and other dementias 0.04246
Cause_Diarrheal diseases                      0.03777
Cause_Drug-susceptible tuberculosis           0.03752
Cause_Colon and rectum cancer                 0.03680
Cause_Neonatal encephalopathy due to birth asphyxia and trauma 0.02714
log_NO2                                       0.02161
log_Population                                0.02161
Cause_Asthma                                  0.02052
Region Name_Western Europe                    0.02023
Cause_Drowning                                0.02001
log_PM25                                      0.01913
Cause_Other neonatal disorders                0.01742
Cause_Pertussis                               0.01685
Region Name_Central Europe          

In [24]:
results_word = results_df.round(3)
results_word

,model,RMSE,R2
0,RandomForest,7.474,0.942
1,XGBoost,11.275,0.868
2,CatBoost,16.662,0.713
3,LinearRegression,22.768,0.463
4,Ridge,22.773,0.463
5,Lasso,22.999,0.452


### Prepare Results for Reporting
This cell rounds the results for presentation and reporting, making the output more readable and suitable for publication.

In [25]:
results_word.to_csv("model_comparison.csv")
print("Table saved as model_comparison.xlsx and model_comparison.csv")

Table saved as model_comparison.xlsx and model_comparison.csv


### Export Results
This cell saves the model comparison results to CSV for further analysis, sharing, or publication.

In [ ]:
# Round results
results_plot = results_df.round(3)

# Create bar plot
fig = px.bar(
    results_plot,
    x=results_plot["model"],
    y="R2",
    title="Model Comparison (R² Score)",
    labels={"R2": "R²", "index": "Model"},
    text="R2",
    range_y=[0, 1],  # R² is between 0 and 1
)

# Rotate x-axis labels
fig.update_layout(
    xaxis=dict(tickangle=45),
    yaxis=dict(range=[0, 1]),
    title=dict(font=dict(size=16)),
    template="plotly_white"
)

# Show chart
fig.show()

### Visualize Model Comparison: R² Scores
This cell creates a bar plot of R² scores for each model, visually comparing their predictive performance.

In [39]:
# Round results
results_plot = results_df.round(3)

# Create figure
fig = go.Figure()

# RMSE as bars
fig.add_trace(
    go.Bar(
        x=results_plot["model"],
        y=results_plot["RMSE"],
        name="RMSE",
        opacity=0.8,
        yaxis="y1"
    )
)

# R² as line
fig.add_trace(
    go.Scatter(
        x=results_plot["model"],
        y=results_plot["R2"],
        name="R²",
        mode="lines+markers",
        marker=dict(color="red", size=8),
        line=dict(color="red", width=2),
        yaxis="y2"
    )
)

# Layout with dual axes
fig.update_layout(
    title="Model Comparison: RMSE vs R²",
    xaxis=dict(title="Model", tickangle=45),
    yaxis=dict(title="RMSE", tickfont=dict(color="blue")),
    yaxis2=dict(
        title="R²",
        tickfont=dict(color="red"),
        overlaying="y",
        side="right",
        range=[0, 1]  # since R² is between 0 and 1
    ),
    legend=dict(x=0.5, y=-0.2, orientation="h", xanchor="center"),
    bargap=0.3,
    template="plotly_white"
)

fig.show()

### Visualize Model Comparison: RMSE vs R²
This cell creates a dual-axis plot showing both RMSE and R² for each model, providing a comprehensive view of predictive accuracy and error.


## Discussion: Model Selection, Results Interpretation, and Random Forest Performance

### Model Selection Defense
A range of models was chosen to balance interpretability, flexibility, and predictive power. Linear models (Linear Regression, Ridge, Lasso) provide a baseline and are easy to interpret, but may miss complex relationships. XGBoost and CatBoost are advanced tree-based models known for handling non-linearities and categorical data. Random Forest, an ensemble of decision trees, is robust to overfitting and effective for tabular data with mixed feature types.

### Results Interpretation
The results table and plots show that Random Forest achieved the best performance (lowest RMSE, highest R²) on the test set. This indicates it captured the underlying patterns in the data more effectively than other models.

### Why Did Random Forest Work Best?
- **Non-linear Relationships:** Random Forest can model complex, non-linear interactions between pollutants, population, and mortality rates, which linear models cannot.
- **Feature Interactions:** It automatically captures interactions between features (e.g., combined effects of multiple pollutants).
- **Robustness to Outliers and Noise:** Random Forest is less sensitive to outliers and noisy data, which are common in real-world environmental datasets.
- **No Need for Extensive Tuning:** Unlike boosting models, Random Forest performs well with default settings and is less prone to overfitting.
- **Interpretability:** Feature importance scores from Random Forest help identify key drivers of mortality.

In summary, Random Forest's flexibility and robustness make it an excellent choice for this predictive task, outperforming both linear and more complex boosting models in this context.